<a href="https://colab.research.google.com/github/parvezshahriar/-Malicious-website-detection-using-machine-and-deep-learning/blob/main/Claude_api_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install antropic python-dotenv

In [ ]:
ANTROPIC_API_KEY = "YOUR_API_KEY"

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from antropic import Antropic
client=Antropic()
model="claude-sonnet-4-5"

In [ ]:
message=client.message.create(
    model=model,
    max_tokens=1000,
    message=[
        {
            "role":"user",
            "content":"whats the weather today in Dhaka?"
        }
    ]
)

In [ ]:
#Multi turn conversation
def add_user_message(message,text):
  user_message={"role": "user", "content":text}
  message.append(user_message)

def add_assistant_message(message,text):
  assistant_message={"role":"assistant","content":text}
  message.append(assistant_message)

def chat(message):
  system_prompt="You are a helpful assistant."
  message=client.message.create(
      model=model,
      max_tokens=1000,
      system=system_prompt,
      message=message
  )
  return message.content[0].text

_IncompleteInputError: incomplete input (2367343084.py, line 10)

In [ ]:
message=[]
add_user_message(message,"Whats the weather today in Dhaka?")
answer=chat(message)
add_assistant_message(message,answer)
add_user_message(message,"Whats the weather today in Chittagong?")
final_answer=chat(message)

In [ ]:
#Building a Flexible Chat Function
def chat(messaeg,system=None):
  params={
      "model":model,
      "max_token":1000,
      "message":message,
  }
  if system:
    param['system']=system

  message=client.message.create(**params)
  return message.content[0].text

In [ ]:
answer=chat(message)

system="""
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

answer=chat(message,system=system)

In [ ]:
#Building a Flexible Chat with Temperature
def chat(messaeg,system=None,temperature=1.0):
  params={
      "model":model,
      "max_token":1000,
      "message":message,
      "temperature":temperature
  }
  if system:
    param['system']=system

  message=client.message.create(**params)
  return message.content[0].text

In [ ]:
answer=chat(message,temperatue=0.0)
answer=chat(message,temperatue=0.5)
answer=chat(message,temperatue=1.0)

In [ ]:
#Response streaming
message=[]
add_user_message(messages,'Write a 1 sentence description of a fake database')
stream=client.message.create(
    model=model
    max_tokens=1000,
    message=message,
    stream=True
)

for event in stream:
  print(event)


##Simplified Text Streaming
with client.message.stream(
    model=model,
    max_tokens=1000,
    message=message
) as stream:
  for text in stream.text_stream:
    pass
  final_message=stream.get_final_message()


In [ ]:
#Structured data
import json
message=[]
add_user_message(message,"Generate a very short event bridge rule as json")
add_assistant_message(message,"```json")

text=chat(message,stop_sequences=["```"])

clean_json=json.loads(text.strip())


In [ ]:
##Generating test datasets
def add_user_message(message,text):
  user_message={"role":"user","contetnt":text}
  message.append(user_message)

def add_assistant_message(message,text):
  assistant_message={"role":"assistant","content":text}
  message.append(assistant_message)

def chat(message,system=None,temperature=1.0,stop_sequences=[]):
  params={
      "model":model,
      "max_tokens":1000,
      "message":message,
      "temperature":temperature,
  }
  if system:
    params["system"]=system
  if stop_sequences:
    params["stop_sequences"]=stop_sequences

  response=client.message.create(**params)
  return response.content[0].text


In [ ]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  \{
    "task": "Description of task",
  \},
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

In [ ]:
message=[]
add_user_message(message,prompt)
add_assistant_message(message,"```json")

text=chat(message,stop_sequences=["```"])
return json.loads(text)

In [ ]:
dataset=generate_dataset()
print(dataset)

In [ ]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [ ]:
##Running the eval
def run_pompt(test_case):
   """Merges the prompt and test case input, then returns the result"""
   prompt=f"""
   please solve the following task:
   {test_case["task"]}
   """
     message=[]
     add_user_message(message,prompt)
     output=chat(message)
     return output


In [ ]:
def run_test_case(test_case):
   """Calls run_prompt, then grades the result"""
   output=run_prompt(test_case)

   score=10

   return {
       "output":output,
       "test_case":test_case,
       "score":score
   }


In [ ]:
def run_eval(dataset):
  """Loads the dataset and calls run_test_case with each case"""
  results=[]

  for test_case in dataset:
    result=run_test_case(test_case)
    result.append(result)

  return results


In [ ]:
with open("dataset.json","r") as f:
  dataset=json.load(f)

result=run_eval(dataset)

print(json.dumps(results,indent=2))

In [ ]:
##Model Grader
def grade_by_model(test_case,output):
  eval_prompt="""
   You are an expert code reviewer. Evaluate this AI-generated solution.

    Task: {task}
    Solution: {solution}

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    message=[]
    add_user_message(message,eval_prompt)
    add_assistant_message(message,"```json")

    eval_text=chat(message,stop_sequences=["```"])
    return json.loads(text)

In [ ]:
##Code based grading
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [ ]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  \{
    "task": "Create a Python function to validate an AWS IAM username",
    "format": "python"
  \},
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

In [ ]:
add_assistant_message(messages, "```code")

In [ ]:
model_grade = grade_by_model(test_case, output)
model_score = model_grade["score"]
syntax_score = grade_syntax(output, test_case)

score = (model_score + syntax_score) / 2

In [ ]:
##Setting Up Your Evaluation Pipeline
evaluator=PromptEvaluator(max_concurrent_tasks=5)

In [ ]:
dataset=evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete"
    },
    output_file="dataset.json",
    num_cases=3
)

In [ ]:
def run_prompt(prompt_inputs):
  prompt=f"""
  What should this person eat?

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""

 messages=[]
 add_user_message(message,prompt)
 output=chat(message)
 return output

In [ ]:
results=evaluator_run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json"
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
"""
)

In [ ]:
#Claude tool function
def get_weather(location):
  if not location or location.strip()=="":
    raise ValueError("Location can not be empty")

  url="https://api.openweathermap.org/data/2.5/weather"
  params={
      "q":location,
      "appid":WEATHER_API_KEY,
      "units":"metric"
  }
  response=requests.get(url,params=params,timeout=10)
  response.raise_for_status()

  weather_data=response.json()
  return

In [ ]:
from antropic.types import ToolParam
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
  if not date_format:
    raise ValueError("date_format cannot be empty")
  return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

In [ ]:
# Default format: "2024-01-15 14:30:25"
get_current_datetime()

# Just hour and minute: "14:30"
get_current_datetime("%H:%M")

In [ ]:
message=[]
message.append({
    "role":"user",
    "content":"Whats the exact time, formatted as HH:MM:SS"
})

response=client.message.create(
    model=model,
    max_tokens=1000,
    message=message,
    tools=[get_current_datetime_schema]
)

In [ ]:
#Multi-turn conversations with tools
def run_conversation(messages):
  while True:
    response=chat(messages)
    add_assistant_message(messages,response)

    if response isn't asking for a tool:
      break

    tool_result_blocks=run_tools(response)
    add_user_message(message,tool_result_blocks)

return messages


In [ ]:
from antropic.types import Message

def add_user_message(messages,message):
  user_message={
      "role":"user",
      "content":message.content if isinstance(message,Message) else message
  }
  messages.append(user_message)


In [ ]:
def chat(messages,system=None,temperature=1.0,stop_sequences=[],tools=None):
  params={
      "model":model,
      "max_tokens":1000,
      "messages":"messages",
      "temperature":temperature,
      "stop_sequences":stop_sequences
   }
   if tools:
    params["tools"]=tools

   if system:
    params["system"]:"system"

    message=client.message.create(**params)
    return message



In [ ]:
def text_from_message(message):
  return "\n".join(
      [block.text for block in message.content if block.type=="text"]
  )

In [ ]:
#Implementing multiple turns
if response.stop_reason!="tool_use":
  break

In [ ]:
def run_conversation(messages):
  while True:
    response=chat(messages,tools=[get_current_datetime_schema])
    add_assistant_message(messages,response)
    print(text_from_message(response))

    if response.stop_reason!="tool_use":
      break

    tool_result_blocks=run_tools(response)
    add_user_message(message,tool_result_blocks)

  return messages

In [ ]:
def run_tools(message):
  tool_requests=[
      block for block in message.content if block.type =="tool_use"
  ]
  tool_result_blocks=[]

  for tool_request in tool_requests:
    if tool_request.name=="get_current_datetime":
      tool_output=get_current_datetime(**tool_request.input)
      tool_result_block={
          "type":"tool_result",
          "tool_use_id":tool_request.id,
          "content":json.dumps(tool_output)
      }



In [ ]:
#error handling for tool output
try:
  tool_output=run_tool(tool_request.name,tool_request.input)
  tool_result_block={
          "type":"tool_result",
          "tool_use_id":tool_request.id,
          "content":json.dumps(tool_output),
          "is_error":False
      }
except Exception as e:
  tool_result_block={
          "type":"tool_result",
          "tool_use_id":tool_request.id,
          "content":f"Error:{e}",
          "is_error":True
      }


In [ ]:
def run_tool(tool_name,tool_input):
  if tool_name=="get_current_datetime":
    return get_current_datetime(**tool_input)
  elif tool_name=="another_tool":
    return another_tool(**tool_input)

In [ ]:
#using multiple tools
response=chat(message,tools=[
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema
    ])

In [ ]:
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)

In [ ]:
#fine grained tool caling
for chunk in stream:
    if chunk.type == "input_json":
        # Process the partial JSON chunk
        print(chunk.partial_json)
        # Or use the complete snapshot so far
        current_args = chunk.snapshot

In [ ]:
run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True
)

In [ ]:
try:
    parsed_args = json.loads(chunk.snapshot)
except json.JSONDecodeError:
    # Handle invalid JSON appropriately
    print("Received invalid JSON, continuing...")

In [ ]:
#the text edit tool
def get_text_edit_schema(model):
  if model.startwith("claude-3-7-sonnet"):
    return{
        "type":"text_editor_20250124",
        "name":"str_replace_editor",
    }
  elif model.startwith("claude-3-5-sonnet"):
    return{
        "type":"text_editor_20241022",
        "name":"str_replace_editor",
    }

In [ ]:
#the web search tool
web_search_schema={
    "type":"web_search_20250305",
    "name":"web_search",
    "max_user":5,
    "allow_domains":["nih.gov"]
}